# Hydro Lens — 100-Epoch Retrain (GPU, D1 + D4 only)

**Model:** YOLOv8n · **Task:** binary object detection (`particle`) · **Epochs:** 100 · **Device:** CUDA (RTX 4060), no CPU fallback

This is the 100-epoch run. The 40-epoch hackathon model stays untouched in
`training/train_40_epochs.ipynb` and `models/best.pt` until this run beats it.

| Item | Value |
|------|-------|
| Dataset | D1 (`sombsuk/dataset_microplastic`) + D4 (`andreusv/Microplastic-Fragments-WWTP-Dataset`) only |
| Splits | 3,709 train / 558 val / 301 test |
| Classes | 1 (`particle`) |
| Image size | 640 · Batch 16 · Seed 42 |
| Epochs | 100, `patience=0` (early stopping disabled → always runs all 100) |
| Baseline (40 ep, keep-if-not-beaten) | test P 0.773 · R 0.776 · mAP50 0.846 · mAP50-95 0.676 |
| Artifacts | `training/hydro_lens_v1_100ep/` |

**Run order:** GPU check → Dataset prep → Train (100 epochs) → Test evaluation → Compare & promote → Curves.
Re-running is safe: dataset prep is skipped if `data/processed/` exists, and the run folder is reused (`exist_ok=True`).

**Promotion rule:** the new model replaces `models/best.pt` only if it is **not worse on either
test mAP50 and mAP50-95, and strictly better on at least one**. Otherwise the 40-epoch model stays deployed.

In [ ]:
from pathlib import Path
import json, shutil, sys

import matplotlib.pyplot as plt
import pandas as pd
import torch
from ultralytics import YOLO

# notebook lives in training/, repo root is its parent
ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

OUT_DIR = ROOT / "training" / "hydro_lens_v1_100ep"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_YAML = ROOT / "data" / "processed" / "data.yaml"
WEIGHTS = OUT_DIR / "weights" / "best.pt"
DEPLOYED = ROOT / "models" / "best.pt"

EPOCHS, BATCH, IMGSZ, SEED, PATIENCE = 100, 16, 640, 42, 0   # patience 0 = no early stop

BASELINE = {  # 40-epoch model currently in models/best.pt (held-out test, 301 images)
    "precision": 0.773, "recall": 0.776, "mAP50": 0.846, "mAP50-95": 0.676,
}

print(f"repo:      {ROOT}")
print(f"artifacts: {OUT_DIR}")
print(f"epochs={EPOCHS} batch={BATCH} imgsz={IMGSZ} seed={SEED} patience={PATIENCE}")

## 0 · GPU check (hard requirement)

This run is GPU-only. If CUDA is not available the cell raises instead of silently
falling back to CPU (a CPU run of 100 epochs would take many hours).

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA not available — this notebook must run on the RTX 4060. "
        "Install a CUDA build: pip install torch==2.14.1 torchvision==0.29.1 "
        "--index-url https://download.pytorch.org/whl/cu126 --force-reinstall"
    )

DEVICE = "cuda"
gpu = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu} ({vram_gb:.1f} GB)  |  torch {torch.__version__}")

## 1 · Dataset preparation (D1 + D4 only)

Builds `data/processed/` (YOLO format, single class `particle`) from the raw D1 zip
archives and D4 parquet files via `src/data/prepare.py`. No other dataset is used.
Skipped when already built.

In [ ]:
if DATA_YAML.exists():
    print(f"dataset already prepared: {DATA_YAML}")
else:
    sys.path.insert(0, str(ROOT))
    from src.data import prepare
    prepare.main()

## 2 · Train — 100 epochs on GPU

`project=training` + `name=hydro_lens_v1_100ep` puts every artifact (weights,
`results.csv`, `args.yaml`, confusion matrix, PR/F1 curves, batch predictions) into
`training/hydro_lens_v1_100ep/`. Nothing else in the repo is touched — in particular
`models/best.pt` is **not** modified here, only in the promotion cell below.

In [ ]:
model = YOLO("yolov8n.pt")

results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    patience=PATIENCE,   # 0 = run all 100 epochs, no early stopping
    batch=BATCH,
    imgsz=IMGSZ,
    seed=SEED,
    device=DEVICE,       # "cuda" only — never cpu/mps
    workers=2,           # 4K source images: 8 workers exhausted 16 GB RAM (cv2 OOM crash)
    project=str(ROOT / "training"),
    name="hydro_lens_v1_100ep",
    exist_ok=True,
    verbose=True,
)

print("saved to", results.save_dir)

## 3 · Held-out test evaluation

Scores the best checkpoint on the 301-image test split (never seen during training)
and freezes the numbers as `test_metrics.json` in the run folder.

In [ ]:
best = YOLO(str(WEIGHTS))
metrics = best.val(data=str(DATA_YAML), split="test", imgsz=IMGSZ, device=DEVICE, workers=2, verbose=True)

new_metrics = {
    "precision": round(float(metrics.box.mp), 4),
    "recall": round(float(metrics.box.mr), 4),
    "mAP50": round(float(metrics.box.map50), 4),
    "mAP50-95": round(float(metrics.box.map), 4),
}

(OUT_DIR / "test_metrics.json").write_text(json.dumps(new_metrics, indent=2))
print(json.dumps(new_metrics, indent=2))

## 4 · Compare with the 40-epoch model & promote

Side-by-side test scores. `REPLACE` = copy the new weights to `models/best.pt`
(old ones are first backed up to `models/best_v1_40ep.pt`).
`KEEP` = the 40-epoch model stays deployed and the new run is left as an artifact.

In [ ]:
rows = []
for k in ("precision", "recall", "mAP50", "mAP50-95"):
    b, n = BASELINE[k], new_metrics[k]
    rows.append({"metric": k, "baseline_40ep": b, "new_100ep": n,
                 "delta": round(n - b, 4), "better": n > b})
cmp_df = pd.DataFrame(rows)
print(cmp_df.to_string(index=False))

# non-regression rule: not worse on mAP50 AND mAP50-95, strictly better on at least one
not_worse = (new_metrics["mAP50"] >= BASELINE["mAP50"] and
             new_metrics["mAP50-95"] >= BASELINE["mAP50-95"])
strictly_better = (new_metrics["mAP50"] > BASELINE["mAP50"] or
                   new_metrics["mAP50-95"] > BASELINE["mAP50-95"])
verdict = "REPLACE" if (not_worse and strictly_better) else "KEEP"

if verdict == "REPLACE":
    backup = DEPLOYED.with_name("best_v1_40ep.pt")
    if DEPLOYED.exists() and not backup.exists():
        shutil.copy2(DEPLOYED, backup)
        print(f"backed up previous model -> {backup}")
    shutil.copy2(WEIGHTS, DEPLOYED)
    print(f"promoted 100-epoch weights -> {DEPLOYED}")
else:
    print(f"kept current model -> {DEPLOYED} (100-epoch run left in {OUT_DIR})")

(OUT_DIR / "comparison.json").write_text(json.dumps({
    "verdict": verdict, "baseline": BASELINE, "new": new_metrics,
}, indent=2))
print("VERDICT:", verdict)

## 5 · Training curves

Rebuilt from `results.csv` so the plots live in the run folder next to the weights.

In [ ]:
df = pd.read_csv(OUT_DIR / "results.csv")
df.columns = [c.strip() for c in df.columns]

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
ax[0].plot(df["epoch"], df["train/box_loss"], label="box")
ax[0].plot(df["epoch"], df["val/box_loss"], label="val/box")
ax[0].set(title="Loss", xlabel="epoch"); ax[0].legend(); ax[0].grid(alpha=.3)

ax[1].plot(df["epoch"], df["metrics/mAP50(B)"], label="mAP50")
ax[1].plot(df["epoch"], df["metrics/mAP50-95(B)"], label="mAP50-95")
ax[1].set(title="Accuracy", xlabel="epoch"); ax[1].legend(); ax[1].grid(alpha=.3)

fig.tight_layout()
fig.savefig(OUT_DIR / "training_curves.png", dpi=150)
plt.show()

print(f"best val mAP50 at epoch {int(df.loc[df['metrics/mAP50(B)'].idxmax(), 'epoch'])}")

## 6 · Artifacts

Everything this run produced, in one folder.

In [ ]:
shutil.copy2(DATA_YAML, OUT_DIR / "data.yaml")   # dataset config snapshot

for p in sorted(OUT_DIR.rglob("*")):
    if p.is_file():
        print(f"{p.relative_to(ROOT)}  ({p.stat().st_size / 1e6:.2f} MB)")

**Deployed weights:** only changed by the promotion cell above.
`REPLACE` → `models/best.pt` = 100-epoch model (40-epoch backup at `models/best_v1_40ep.pt`).
`KEEP` → `models/best.pt` unchanged (40-epoch model).